<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/01.RR_Respiratory_Mortality_AirQ%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👩‍💻 Estimation of Relative Risk of Respiratory Mortality Attributable to Long-Term PM₂.₅ Exposure in Argentina (2022) Using the AirQ+ Methodology


Long-term exposure to ambient fine particulate matter with an aerodynamic diameter ≤ 2.5 µm (PM₂.₅) is a well-established risk factor for premature mortality due to respiratory diseases. Quantifying the health burden attributable to PM₂.₅ exposure is essential for evidence-based public health decision-making and environmental policy evaluation. The World Health Organization (WHO) has developed the AirQ+ software as a standardized tool to estimate the population health impacts of air pollution using concentration–response functions derived from epidemiological studies.[1]

The objective of this analysis is to estimate Relative Risk (RR) of Respiratory Mortality Attributable to Long-Term PM₂.₅ in Argentina for the year 2022, in those department where actually occured asthma mortality events. Following the methodology implemented in AirQ+,the health endpoint “mortality, respiratory, adults aged 30 years and older” will be used. [1]

## 📚 Import required libraries

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd
import mapclassify

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
# from matplotlib.patches import
import matplotlib.patches as mpatches
import seaborn as sns

# other libraries
import os
from itables import init_notebook_mode, show

In [ ]:
# change to working directory
%cd work/

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

##🗂 Load and prepare the dataset

In [ ]:
# open results_RF.gpkg as a gdf
gdf_rf_results = gpd.read_file("pdt/asthma_mortality/data/gpkg/results_RF.gpkg")

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=False)
gdf_rf_results.head()

Column names:
* IDDPTO: ID of the department
* AMR_2022: Asthma Mortality Rate (AMR) per 100,000 inhabitants in 2022
* AMR_2022_PRED: Predicted AMR per 100,000 inhabitants in 2022

In [ ]:
# Load dataset with data per department
gdf_data = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
gdf_data.head()

In [ ]:
# add PM25_2022 (mean PM2.5 concentration µg/m³)
# from gdf_data to gdf_rf_results using IDDPTO as join column
gdf = gdf_rf_results.merge(gdf_data[['IDDPTO', "PM25_2022"]], on='IDDPTO', how='left')

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
gdf.head()

Following the hurdle-type Random Forest modeling, we will select data only for those departments where asthma mortality actually occurred in 2022.


In [ ]:
# filter out departments where AMR_2022 (Asthma Mortality Rate) is greater than 0
gdf = gdf[gdf['AMR_2022'] != 0]

In [ ]:
# drop geometry column
df = gdf.drop(columns=['geometry'])

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
show(df)

##  😮‍💨🫁 Relative Risk of Respiratory Mortality Associated with Long-Term Exposure to PM₂.₅

In epidemiology, RR quantifies the strength of the association between exposure to a risk factor and a health outcome. It is defined as the ratio of the risk of the outcome in the exposed population to the risk in the unexposed (or less exposed) population. An RR greater than 1 indicates an increased risk associated with exposure, while an RR equal to 1 indicates no association. [2]

In the AirQ+ framework, the association between long-term exposure to PM₂.₅ and mortality is modeled using a log-linear concentration–response function, derived from large cohort studies. This approach assumes that the logarithm of the relative risk increases linearly with increasing pollutant concentration above a specified counterfactual or cut-off value.

Following the AirQ+ methodology, the RR of respiratory mortality associated with long-term PM₂.₅ exposure is calculated using the equation:

\begin{align*}
RR (X) = e^{[\beta (X - X_0)]}
\end{align*}


where $X$ is the annual mean $PM_{2.5}$ concentration, $X_0$ is the counterfactual (cut-off) concentration, and $\beta$ is the concentration–response coefficient. In this analysis, a cut-off value of $X_0 = 10\mu$g/$m^3$ is used, in line with the AirQ+ default settings. RRs are calculated using the central estimate of $\beta$, as well as its lower and upper bounds, in order to reflect uncertainty in the exposure–response relationship.



In [ ]:
# Define AirQ+ concentration–response coefficients (beta)
# Central estimate of beta
beta_central = 0.009531017980432493

# Lower bound of beta (uncertainty interval)
beta_lower = 0.002955880224154443

# Upper bound of beta (uncertainty interval)
beta_upper = 0.016551443847757332

In [ ]:
# Define counterfactual (cut-off) concentration X0
# Cut-off concentration for PM2.5 (µg/m³), as used in AirQ+
X0 = 10

In [ ]:
# Calculate the RR of Respiratory Mortality (RRRM) for each department

# Relative Risk using central beta
df["RRRM_2022_C"] = round(np.exp(beta_central * (df["PM25_2022"] - X0)),3)

# Relative Risk using lower beta
df["RRRM_2022_L"] = round(np.exp(beta_lower * (df["PM25_2022"] - X0)),3)

# Relative Risk using upper beta
df["RRRM_2022_U"] = round(np.exp(beta_upper * (df["PM25_2022"] - X0)),3)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
df.head()

In department 06588, an estimated RR  of 1.041 indicates that, at an annual mean PM₂.₅ concentration of 14.24 µg/m³, the RR of respiratory mortality is 4.1% higher compared with the counterfactual concentration of 10 µg/m³ of $PM_{2.5}$.

## 🗺️  Mapping RR of Respiratory Mortality Associated with Long-Term Exposure to PM₂.₅

We will map the RR of respiratory mortality (lower, central, and upper bounds) for 2022 only in departments where mortality actually occurred during that year. Departments where asthma mortality did not occur in 2022 are shown as hollow polygons with a thin black border.

### Selecting data for mapping

In [ ]:
# select "IDDPTO" and "geometry" from gdf_data
gdf_map = gdf_data[["IDDPTO", "geometry"]]

In [ ]:
# Merge df with gdf_map
gdf_map_rr = gdf_map.merge(
    df,
    on='IDDPTO',
    how='left'
)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_map_rr.head()

In [ ]:
# drop all rows containing NaN for all rows
gdf_map_rr_valid = gdf_map_rr.dropna().reset_index(drop=True)

In [ ]:
len(gdf_map_rr_valid)

In [ ]:
# select all NaN rows from gdf_map_rr
gdf_map_rr_nan = gdf_map_rr[gdf_map_rr.isna().any(axis=1)]

In [ ]:
len(gdf_map_rr_nan)

### Calculate classification schema for mapping

We will use [Pysal](https://pysal.org/)’s [mapclassify](https://pysal.org/mapclassify/index.html) library to identify the most appropriate classification scheme for the choropleth map. The selected classifier will be the one with the lowest ACDM (Mean Absolute Deviation around the Class Median). In this context, ACDM measures how well a classification fits the data by calculating the average absolute distance between each observation and the median of its assigned class.

In [ ]:
# Select columns corresponding to RR
selected_data_RR = gdf_map_rr_valid.loc[:,["RRRM_2022_C", "RRRM_2022_L", "RRRM_2022_U"]]

In [ ]:
# Classify the data into 5 quantile groups
q5 = mapclassify.Quantiles(selected_data_RR, k=5)
q5

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data_RR, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data_RR)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data_RR, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data_RR)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data_RR, k=5)
fj5

In [ ]:
# get interval labels from Fisher-Jenks classification
min_val = selected_data_RR.min().min()

interval_labels = []
current_lower_bound = min_val

for upper_bound in fj5.bins:
    label = f"{current_lower_bound:.2f}, {upper_bound:.2f}"
    interval_labels.append(label)
    current_lower_bound = upper_bound

print(interval_labels)

In [ ]:
# Bunch classifier objects
class5 = q5, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)

Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

### Create choropleth maps

In [ ]:
# Convert the bins to a list for further processing
bins = fj5.bins.tolist()
bins

In [ ]:
# Create a custom classification using UserDefined() for RR Respiratory Mortality lower bound
classi_rr_l = mapclassify.UserDefined(gdf_map_rr_valid["RRRM_2022_L"], bins)

In [ ]:
# Create a custom classification using UserDefined() for RR Respiratory Mortality central bould
classi_rr_c = mapclassify.UserDefined(gdf_map_rr_valid["RRRM_2022_C"], bins)

In [ ]:
# Create a custom classification using UserDefined() for RR Respiratory Mortality upper bould
classi_rr_u = mapclassify.UserDefined(gdf_map_rr_valid["RRRM_2022_U"], bins)

In [ ]:
# get interval labels from Fisher-Jenks classification
min_val = selected_data_RR.min().min()

interval_labels = []
current_lower_bound = min_val

for upper_bound in fj5.bins:
    label = f"{current_lower_bound:.2f}-{upper_bound:.2f}"
    interval_labels.append(label)
    current_lower_bound = upper_bound

interval_labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 7))
fig.subplots_adjust(hspace=0, wspace=-0.5)
plt.suptitle('RR of Respiratory Mortality Associated with Long-Term Exposure to PM₂.₅ 2022', fontsize=14, y=1)

# maps
for ax, classi in zip(axes, [classi_rr_l, classi_rr_c, classi_rr_u]):
    classi.plot(
        gdf_map_rr_valid,
        legend=False,
        axis_on=False,
        border_color='black',
        cmap="viridis_r",
        ax=ax
    )
    gdf_map_rr_nan.plot(ax=ax, color="none", edgecolor="black", linewidth=0.10)

# cuntom legend (bins)
bin_labels = interval_labels
n_bins = len(bin_labels)
cmap = mpl.colormaps.get_cmap("viridis_r").resampled(n_bins)
colors = [mpl.colors.to_hex(cmap(i)) for i in range(cmap.N)]

bin_patches = [
    mpatches.Patch(facecolor=color, edgecolor='black', label=label)
    for color, label in zip(colors, bin_labels)
]

# "No Data" patch
no_data_patch = mpatches.Patch(
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    label='No Data'
)

# legend
fig.legend(
    handles=bin_patches,
    loc='lower center',
    bbox_to_anchor=(0.4, -0.02),
    ncol=len(bin_patches),
    fontsize=9,
    frameon=False
)

# "No data" legend
fig.legend(
    handles=[no_data_patch],
    loc='lower center',
    bbox_to_anchor=(0.75, -0.02),
    fontsize=9,
    frameon=False
)

# titles
axes[0].set_title('RR Respiratory Mortality \n Lower Bound', fontsize=12)
axes[1].set_title('RR Respiratory Mortality \n Central Bound', fontsize=12)
axes[2].set_title('RR Respiratory Mortality \n Upper Bound', fontsize=12)

plt.tight_layout()
plt.show()

In 2022, long-term exposure to PM₂.₅ was associated with an increased RR of respiratory mortality across Argentine departments, although the magnitude varied spatially and across uncertainty bounds. The lower-bound estimates indicate relatively low and homogeneous risk in most departments where asthma mortality occurred. However, the central estimates show relative risks mainly in the range of 1.08–1.16, suggesting that PM₂.₅ exposure is associated with an 8–16% higher risk of respiratory mortality compared to the reference level.

The upper-bound estimates reveal more pronounced spatial heterogeneity, with the highest relative risks (1.32–1.48) concentrated in  some departments, particularly in regions such as Patagonia and Cuyo. These elevated values likely reflect areas with higher PM₂.₅ exposure.

Overall, the maps highlight a spatially uneven distribution of the respiratory health burden attributable to PM₂.₅ in Argentina, with certain departments experiencing substantially higher risk than others.

In [ ]:
# save gdf_map_rr as gpkg
#gdf_map_rr.to_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg", driver="GPKG")

## 📚 References

[1] WHO Regional Office for Europe, European Centre for Environment and Health (2022). AirQ+: software tool for health risk assessment of air pollution. Bonn (Germany): WHO Regional Office for Europe.
https://www.who.int/europe/tools-and-toolkits/airq---software-tool-for-health-risk-assessment-of-air-pollution
Accessed 01/2026.

[2] J. Zhang and K. F. Yu, “What’s the Relative Risk?A Method of Correcting the Odds Ratio in Cohort Studies of Common Outcomes,” JAMA, vol. 280, no. 19, pp. 1690–1691, Nov. 1998, doi: 10.1001/jama.280.19.1690.
